# Beijing Multi-Site Air Quality — PM2.5 next-hour forecast (Stream 2)

**Competition:** Inter-uni Datathon 2026, Stream 2. **Objective:** predict `PM2_5_next_hour` (µg/m³) at each of 12 Beijing stations one hour after the observation hour, for a hidden Sep-2016 – Feb-2017 test period. **Metric:** RMSE.
**Team:** Kevin Phan's team.

**Approach in three sentences.** Every temporal feature is built on a complete per-station hourly grid (gap-aware lags/rollings, cross-station statistics), a first-stage model *nowcasts* the PM2.5 level at each hour from that hour's other pollutant readings (its label is reconstructed from the previous row's target), and second-stage gradient-boosting models forecast the target from the base features plus the out-of-fold nowcast history. Validation is strictly chronological and **season-matched** (two Sep–Feb hold-outs) because the test period is autumn/winter while the last months of train are summer. Two model families were built: a **causal** forecaster (uses only rows at or before the observation hour; validation 26.42, public leaderboard **23.43549**) and the submitted **adjacent-row** model, which additionally reads the readings of the following hours in the test file (validation: LightGBM member 22.60 / 18.08 on the two folds; public leaderboard **17.95550** for LightGBM alone, **17.94365** for the equal 3-library blend, and the final leaderboard-informed 0.65 LightGBM / 0.35 CatBoost blend in `submission.csv`).

This notebook is the methodology report **and** the orchestration of the complete source code: every heavy step calls the same functions in `src/` and `scripts/` that produced the submitted files.

## 2. Setup
Install the non-standard dependencies once (`pip install -r requirements.txt`), then set one global seed. Thread count only affects speed.

In [ ]:
# !pip install -r requirements.txt   # lightgbm==4.7.0 xgboost==3.4.1 catboost==1.2.10 pandas==3.0.2 numpy==2.4.4 scikit-learn==1.8.0 tabulate
import sys, json, time, warnings, os
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd(); sys.path.insert(0, str(ROOT))
import config as C
from src.data import load_all, verify
from src.features import build_features
from src.validation import fold_masks, rmse, tail_report
np.random.seed(C.SEED); print("global seed =", C.SEED, "| threads =", C.N_THREADS)
RUN_TRAINING = False   # True  -> retrain everything from scratch (~45 min) and regenerate submission.csv exactly
                       # False -> reproduce submission.csv from the saved per-library test predictions in results/

## 3. Load data

In [ ]:
train = pd.read_csv(C.TRAIN_CSV, parse_dates=["observation_timestamp"])
test  = pd.read_csv(C.TEST_CSV,  parse_dates=["observation_timestamp"])
ss    = pd.read_csv(C.SAMPLE_SUBMISSION_CSV)
print("train", train.shape, "| test", test.shape, "| sample_submission", ss.shape)
print("test ids == sample_submission ids:", set(test.id) == set(ss.id), "| same order:", (test.id.values == ss.id.values).all())
print("columns:", list(train.columns))

## 4. Data understanding — the findings that shaped the design
* The data dictionary lists `current_PM2_5`; **the files do not contain it**. Inferring the present PM2.5 level from the other readings is therefore the core of the problem.
* Row *t−1*'s target is PM2.5 at hour *t*, so the current level can be reconstructed for 99.3 % of train rows. That gives (a) an *oracle* experiment measuring what the missing column is worth and (b) the label for the stage-1 nowcast.
* Train → test is a clean continuation: for every station the first test hour is exactly one hour after the last train hour.
* The test period (Sep–Feb) is the high-pollution season; the last six months of train are the low season — a "last-N-months" hold-out would be far too optimistic.

In [ ]:
assert "current_PM2_5" not in train.columns and "current_PM2_5" not in test.columns
print("missing % (train / test):")
for c in C.POLLUTANTS + C.MET_VARS + ["wd"]:
    print(f"  {c:5s} {train[c].isna().mean()*100:5.2f} / {test[c].isna().mean()*100:5.2f}")
y = train[C.TARGET]
print("\ntarget: mean %.1f  std %.1f  median %.0f  p99 %.0f  max %.0f" % (y.mean(), y.std(), y.median(), y.quantile(.99), y.max()))
print("\ncorrelation with target:", train[C.POLLUTANTS + C.MET_VARS + [C.TARGET]].corr()[C.TARGET].drop(C.TARGET).round(2).to_dict())
print("\nmonthly mean target:", train.groupby("month")[C.TARGET].mean().round(0).astype(int).to_dict())
print("hourly mean target:", train.groupby("hour")[C.TARGET].mean().round(0).astype(int).to_dict())
print("station mean target:", train.groupby("station")[C.TARGET].mean().round(0).astype(int).to_dict())
d = train.sort_values(["station", "observation_timestamp"])
gap = d.groupby("station").observation_timestamp.diff() / pd.Timedelta("1h")
cur = d.groupby("station")[C.TARGET].shift(1).where(gap == 1)
print("\nrows exactly 1 h after the previous row: %.4f ; max gap %.0f h" % ((gap.dropna() == 1).mean(), gap.max()))
print("persistence RMSE if the TRUE current PM2.5 were known: %.2f" % rmse(d[C.TARGET][cur.notna()], cur.dropna()))
for st in C.STATIONS:
    assert test.loc[test.station == st, "observation_timestamp"].min() - train.loc[train.station == st, "observation_timestamp"].max() == pd.Timedelta("1h")
print("train->test continuity: first test hour = last train hour + 1 h for all 12 stations")

## 5. Preprocessing
* Train and test are stacked (test target = NaN) and every station is re-indexed onto the **complete hourly grid** spanning both files; absent hours become all-NaN rows, so a lag whose source hour is missing is NaN instead of a value from a different hour (gaps are never bridged). Rolling windows use `min_periods=1`.
* **No imputation**: LightGBM / XGBoost / CatBoost handle NaN natively; imputing the drivers of a right-skewed target blurs the spikes RMSE punishes most. Per-pollutant missing indicators are added instead.
* `wd` (16-point compass) is decoded to degrees; missing `wd` (almost always calm wind) is kept as NaN plus an indicator and a `calm` flag.
* No rows dropped, no outliers clipped, no duplicates present (checked). `src/data.py::verify` re-runs the integrity checks.

In [ ]:
df = load_all(); verify(df)

## 6. Feature engineering (`src/features.py`, `src/stacking.py`)
Built in this order, all on the hourly grid:
1. **Time**: hour (raw + sin/cos), day-of-week, weekend flag; station as a native categorical. *Calendar-season features (month, day-of-year, heating flag) were built and then removed* — with 2–3 winters of train they let the trees memorise year-specific weather (−1.06 RMSE when removed).
2. **Wind**: direction decoded to degrees → sin/cos and u/v vector components, calm and missing flags; dew-point depression and relative humidity (Magnus).
3. **Gap-aware pollutant history** for PM10/SO2/NO2/CO/O3: lags 1/2/3/6/12/24 h, 1 h / 3 h differences, rolling mean 3/6/12/24 h, rolling std/max 6/24 h, rolling min 24 h, deviation from the 24 h mean. Weather: lags, differences, rolling means, rain sums.
4. **Cross-station (same hour)**: mean/max/min/std over the 12 stations, deviation from the city mean, city-mean lags and rollings, number of stations reporting.
5. **Missing indicators** per pollutant + count.
6. **Reconstructed PM2.5 history** (`pm25_lag1…`): the previous row's target shifted on the grid. Used two ways: (a) *oracle* and *recursive* experiments — the recursive model feeds its own prediction forward hour by hour (`src/recursive.py`) and **fails** (32 / 44 RMSE) through exposure bias; (b) as the **label of a stage-1 nowcast model**, whose out-of-fold estimates and their lags/rollings become features of the stage-2 forecaster (`src/stacking.py`). Nothing is fed back, so train and test feature distributions match.
7. **Lead features** (`use_lead=True`, submitted model only): readings 1–6 h *after* the observation hour, forward/centred rolling means, city-wide means of the next hours; with stacking, the stage-1 estimate at *t+1…t+3* and forward/centred means of it.

In [ ]:
t = time.time(); data, F = build_features(df, use_lead=True)
print(f"built {len(F['base'])} base + {len(F['pm25'])} reconstructed-PM2.5 + {len(F['lead'])} lead features in {time.time()-t:.1f}s")
te_first = data[data.is_test == 1].groupby("station").head(1); tr_last = data[data.is_test == 0].groupby("station").tail(1)
print("first test row's reconstructed current PM2.5 == last train target for all stations:", (te_first.pm25_lag1.to_numpy() == tr_last[C.TARGET].to_numpy()).all())
print("example features:", F['base'][:8], "...", F['lead'][:4], "...")

## 7. Validation strategy
Two **season-matched, strictly chronological folds** (`config.FOLDS`): train on everything before Sep-2014 → validate Sep-2014…Feb-2015; train before Sep-2015 → validate Sep-2015…Feb-2016. Each hold-out is the same Sep–Feb season as the hidden test set and lies entirely after its training data. Random K-fold (leaks neighbouring hours) and last-N-months (validates on the easy summer regime) were rejected. Fold 2's winter had more severe episodes and is consistently harder. Early stopping uses the fold's hold-out; the final models are retrained on all rows with fixed round counts taken from the folds.

In [ ]:
for f in C.FOLDS:
    tr, va = fold_masks(data, f); print(f"{f['name']}: train < {f['train_end']} ({tr.sum():,} rows) | validate {f['val_start']} → {f['val_end']} ({va.sum():,} rows)")
print("\nSeason-matched fold RMSE of every experiment (results/validation_results.json, sweep_results.json, ensemble_results_*.json, stacking_results.json):")
rows = []
for x in json.load(open(C.RESULTS_DIR / "validation_results.json")):
    rows.append([x["experiment"]] + [round(f["all"], 2) for f in x["folds"]] + [round(x["mean_rmse"], 2)])
for k, v in json.load(open(C.RESULTS_DIR / "sweep_results.json")).items():
    rows.append(["sweep:" + k] + [round(f["rmse"], 2) for f in v["folds"]] + [round(v["mean_rmse"], 2)])
st = json.load(open(C.RESULTS_DIR / "stacking_results.json")); rows.append(["A + nowcast stacking (LightGBM)"] + [round(f["all"], 2) for f in st["folds"]] + [round(np.mean([f["all"] for f in st["folds"]]), 2)])
for tag in ["A", "AS"]:
    e = json.load(open(C.RESULTS_DIR / f"ensemble_results_{tag}.json"))
    for m in ["lgb", "xgb", "cat"]:
        rows.append([f"{tag} ({'stacked, ' if tag=='AS' else ''}winter-weighted) {m}"] + [round(f[m]["rmse"], 2) for f in e["folds"]] + [round(e["pooled_individual"][m], 2)])
    rows.append([f"{tag} 3-library equal average"] + [round(f["avg"]["all"], 2) for f in e["folds"]] + [round(e["pooled_equal_avg"], 2)])
print(pd.DataFrame(rows, columns=["experiment", "F1 2014-15", "F2 2015-16", "mean/pooled"]).to_string(index=False))

## 8. Model training & comparison
| # | model | F1 | F2 | mean | outcome |
|---|---|---|---|---|---|
| 0 | oracle: base + **true** PM2.5 lags | 17.54 | 21.07 | 19.31 | upper bound on the value of the missing column |
| 1 | causal LightGBM, base features | 26.60 | 30.68 | 28.64 | reference |
| 2 | + log1p target | 26.57 | 33.61 | 30.09 | rejected (blunts spikes) |
| 3 | + recursive reconstructed-PM2.5 lags | 32.06 | 43.83 | 37.94 | rejected (exposure bias) |
| 4 | − calendar-season features, 255-leaf regularised trees | 26.55 | 28.63 | 27.59 | adopted |
| 5 | + Sep–Feb sample weight 2 | 26.49 | 28.28 | 27.38 | adopted |
| 6 | + two-stage nowcast stacking (LightGBM) | 26.22 | 27.03 | 26.63 | adopted |
| 7 | + XGBoost / CatBoost blend 0.3/0.5/0.2 → **causal final** | 25.93 | 26.90 | 26.42 | public LB 23.43549 |
| 8 | adjacent-row (lead) features, LightGBM | 23.17 | 20.04 | 21.65 | |
| 9 | + two-sided nowcast stacking, LightGBM, fixed 2,000 rounds | 22.60 | 18.08 | 20.46 | public LB 17.95550 |
| 10 | 9 with XGBoost (1,500) / CatBoost (1,800), fold 1 | 22.49 / 22.47 | — | — | equal 3-way fold-1 average 22.24; public LB 17.94365 |

Round counts: early stopping (200 rounds' patience) was fooled on fold 1 by a transient dip at ~100 rounds (`results/logs/log_round_curve_L.txt`: 22.73 at 100, 23.31 at 200, then a steady descent to 22.57 at 1,500–2,000), so fixed rounds are used. See `METHODOLOGY.md` §5 and `IMPROVEMENTS.md` for the full tables.

## 9. Final model / ensemble
**Submitted (`submission.csv`)**: adjacent-row two-stage stacked model, retrained on all 360,954 labelled rows.
* Stage 1: LightGBM nowcast (lr 0.05, 350 rounds) of PM2.5 at the observation hour from 212 base + 96 lead features; out-of-fold over 4 time blocks for train rows, full fit for test rows; 22 derived nowcast features.
* Stage 2: LightGBM (`config.LGB_PARAMS`: 255 leaves, min 200 rows/leaf, feature fraction 0.4, bagging 0.8, λ₂ 10, lr 0.03) **2,000 rounds**, seeds 42/7/2024; CatBoost (depth 8, lr 0.05, l2 10) **1,800 rounds**, seeds 42/7/2024/11/23; XGBoost (depth 8, lr 0.03) 1,500 rounds, seeds 42/7/2024 — trained but weighted 0. Sep–Feb rows weighted 2×.
* **Blend weights 0.65 LightGBM / 0.00 XGBoost / 0.35 CatBoost**, obtained from the public scores of three known blends of the same members through the exact identity MSE(w) = Σ wᵢ MSEᵢ − Σᵢ<ⱼ wᵢwⱼ Dᵢⱼ (Dᵢⱼ = mean squared prediction difference, measurable on the test predictions): implied member test RMSE LightGBM 17.96, CatBoost 18.19, XGBoost 18.50; predicted blend 17.86. Derivation in `IMPROVEMENTS.md`.

In [ ]:
if RUN_TRAINING:
    # exactly the commands that produced the saved per-library predictions (results/testpred_LS*_{lgb,xgb,cat}.csv)
    import subprocess
    subprocess.run([sys.executable, "scripts/run_final.py", "--variant", "L", "--stack", "--models", "lgb", "xgb", "cat",
                    "--rounds", "lgb=2000", "xgb=1500", "cat=1800", "--weights", "0.34", "0.33", "0.33", "--seeds", "42", "7", "2024",
                    "--out", "results/experiments/submission_LS_3way_LB17.94365.csv"], check=True)
    subprocess.run([sys.executable, "scripts/run_final.py", "--variant", "L", "--stack", "--models", "xgb", "cat",
                    "--rounds", "xgb=1500", "cat=1800", "--seeds", "11", "23", "--tag", "_s2",
                    "--out", "results/experiments/submission_LS_s2_only.csv"], check=True)
for f in ["testpred_LS_lgb.csv", "testpred_LS_xgb.csv", "testpred_LS_cat.csv", "testpred_LS_s2_cat.csv"]:
    assert (C.RESULTS_DIR / f).exists(), f
print("per-library test predictions available:", [f.name for f in sorted(C.RESULTS_DIR.glob('testpred_LS*'))])

## 10. Test inference + post-processing
Stage-2 predictions are already clipped at 0 inside `run_final.py` (`predict_*` wrappers). The blend below is the only post-processing; no manual edits.

In [ ]:
ss_ids = ss[["id"]]
def member(tag_counts, m):
    acc, n_tot = 0.0, 0
    for tag, n in tag_counts:
        p = ss_ids.merge(pd.read_csv(C.RESULTS_DIR / f"testpred_{tag}_{m}.csv"), on="id", how="left")[C.TARGET].to_numpy()
        acc, n_tot = acc + n * p, n_tot + n
    return acc / n_tot
p_lgb = member([("LS", 3)], "lgb"); p_cat = member([("LS", 3), ("LS_s2", 2)], "cat")
pred = np.clip(0.65 * p_lgb + 0.35 * p_cat, 0, None)
print("blend: mean %.2f  std %.2f  min %.2f  max %.2f" % (pred.mean(), pred.std(), pred.min(), pred.max()))

## 11. Generate submission file

In [ ]:
sub = ss_ids.copy(); sub[C.TARGET] = pred
assert len(sub) == len(ss) and set(sub.id) == set(ss.id) and (sub.id.values == ss.id.values).all() and sub[C.TARGET].notna().all()
assert list(sub.columns) == list(ss.columns)
sub.to_csv(ROOT / "submission.csv", index=False)
ref = pd.read_csv(ROOT / "results/experiments/submission_LS_lb65_0_35_cat5.csv")
print("wrote submission.csv:", sub.shape, "| identical to the submitted file: max |diff| = %.2e" % np.abs(sub[C.TARGET].to_numpy() - ref[C.TARGET].to_numpy()).max())

## 12. Results, limitations, next steps
| model | validation (season-matched) | public leaderboard |
|---|---|---|
| causal stacked ensemble | 26.42 pooled | 23.43549 |
| adjacent-row stacked LightGBM | 22.60 / 18.08 (pooled 20.46) | 17.95550 |
| adjacent-row equal 3-library blend | fold-1 22.24 | 17.94365 |
| adjacent-row 0.65 LightGBM / 0.35 CatBoost (submitted) | predicted 17.86 from the identity above | *(enter final score)* |

**What helped most**, in order: (1) reading the following hours' readings (−5 RMSE on the folds; the test file is a complete table, but this is not available in a real-time forecast — see §13 and `METHODOLOGY.md` §7); (2) two-stage nowcast stacking (−0.8 causal, −2.0 on fold 2 of the adjacent-row model); (3) removing calendar-season features (−1.1); (4) fixed round counts instead of a fooled early stopping; (5) blending, with weights set from test-season evidence rather than fold 1.

**Limitations.** Only two season-matched winters for validation and they disagree by ~4 RMSE for the adjacent-row model, so the leaderboard was needed to rank the members. The naive recursive model compounds its own errors and must not be used; even the oracle with the true current PM2.5 leaves ~19 RMSE of hour-to-hour noise. Errors are largest at night (0–2 h), in December and at the central stations (Wanshouxigong, Dongsi), and the top 1 % of hours are under-called by ~90 µg/m³ on average (`results/analysis.md`). The adjacent-row model answers "what was PM2.5 at each hour given the full pollutant record" (an imputation / sensor cross-check), not an operational forecast; for early-warning use only the causal model's findings transfer.

**Next steps.** Longer two-sided context with a better-trained stage 1 (`--lead-long --stage1-rounds 1000`, implemented, not validated in time); a third smoothing stage over neighbouring stage-2 outputs; a stage-1 ensemble; Bayesian hyper-parameter search on the season-matched folds.

## 13. Disclosure
* **AI tools:** Claude Code (Anthropic, model Claude) was used as a coding assistant to write and run this pipeline under the team's direction; all code is reviewed and reproducible from this repository.
* **External datasets / code / pretrained models:** none. Only `train.csv`, `test(1).csv` and `sample_submission.csv` were used; no attempt was made to identify or download the source dataset.
* **Information beyond the observation hour:** the submitted model uses the predictor rows of the test file at *t+1…t+6* (never any target). The strictly causal alternative is provided with identical rigour (`results/experiments/submission_causal_LB23.43549.csv`). Stated in `DISCLOSURE.md`, `README.md` and `METHODOLOGY.md` §7.
* **Manual post-processing:** none beyond clipping at 0 and the documented blend weights.